In [3]:
import tkinter as tk
from tkinter import filedialog
from PIL import Image, ImageTk
import numpy as np
import cv2
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt

# Load trained model
model = load_model("unet_pe_model.h5")   # <-- change path if needed

IMG_SIZE = 256

# Create window
root = tk.Tk()
root.title("Pulmonary Embolism Detection")
root.geometry("800x500")

# Labels to display images
panel_input = tk.Label(root)
panel_input.pack(side="left", padx=10, pady=10)

panel_output = tk.Label(root)
panel_output.pack(side="right", padx=10, pady=10)


# Function to process image
def predict_image(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

    # Resize
    img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img_norm = img_resized / 255.0
    #img_input = np.expand_dims(img_norm, axis=(0, -1))

    img_input = np.array(img_norm).reshape(-1, IMG_SIZE, IMG_SIZE, 1)

    # Prediction
    #pred = model.predict(img_input)
        # U-Net prediction
    pred = model.predict(img_input)[0].squeeze()

    # Convert to binary mask
    mask = (pred > 0.2).astype(np.uint8) * 255

    # Convert original to color for overlay
    img_color = cv2.cvtColor(img_resized, cv2.COLOR_GRAY2BGR)

    # 🔴 Overlay RED mask
    red_mask = np.zeros_like(img_color)
    red_mask[:, :, 2] = mask  # Red channel

    overlay = cv2.addWeighted(img_color, 0.7, red_mask, 0.3, 0)

    # Draw bounding boxes
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        cv2.rectangle(overlay, (x, y), (x+w, y+h), (0, 255, 0), 2)


    plt.figure(figsize=(12,4))
    
    plt.subplot(1,3,1)
    plt.title("CT Image")
    plt.imshow(img_input.squeeze(), cmap='gray')
    
    #plt.subplot(1,3,3)
    plt.subplot(1,3,2)
    plt.title("Predicted Mask")
    plt.imshow(pred, cmap='gray')

    plt.subplot(1,3,3)
    plt.title("PE Mask")
    plt.imshow(overlay)

    
    plt.show()

    #return img_resized, pred[0].squeeze()


# Browse function
def browse_file():
    file_path = filedialog.askopenfilename()

    if file_path:
        predict_image(file_path)
        
        #input_img, output_img = predict_image(file_path)

        '''# Convert to PIL format
        input_pil = Image.fromarray(input_img)
        
        #output_pil = Image.fromarray(output_img)
        overlay = cv2.addWeighted(input_img, 0.7, output_img, 0.3, 0)
        output_pil = Image.fromarray(overlay)

        # Resize for display
        input_pil = input_pil.resize((300, 300))
        output_pil = output_pil.resize((300, 300))

        input_tk = ImageTk.PhotoImage(input_pil)
        output_tk = ImageTk.PhotoImage(output_pil)

        panel_input.config(image=input_tk)
        panel_input.image = input_tk

        panel_output.config(image=output_tk)
        panel_output.image = output_tk'''


# Button
btn = tk.Button(root, text="Browse CT Image", command=browse_file)
btn.pack(pady=20)

root.mainloop()


In [2]:
import tkinter as tk
from tkinter import filedialog
from PIL import Image, ImageTk
import numpy as np
import cv2
from tensorflow.keras.models import load_model

# Load model
model = load_model("unet_pe_model.h5")

IMG_SIZE = 256

# GUI
root = tk.Tk()
root.title("Pulmonary Embolism Detection")
root.geometry("900x500")

panel_input = tk.Label(root)
panel_input.pack(side="left", padx=10)

panel_output = tk.Label(root)
panel_output.pack(side="right", padx=10)

info_label = tk.Label(root, text="", font=("Arial", 12))
info_label.pack(pady=10)


# 🔍 Feature Extraction Function
def extract_features(binary_mask):
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    features_text = ""

    for cnt in contours:
        area = cv2.contourArea(cnt)
        perimeter = cv2.arcLength(cnt, True)

        # Bounding box
        x, y, w, h = cv2.boundingRect(cnt)

        # Centroid
        M = cv2.moments(cnt)
        if M["m00"] != 0:
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
        else:
            cx, cy = 0, 0

        features_text += f"Area: {area:.2f}, Perimeter: {perimeter:.2f}\n"
        features_text += f"Centroid: ({cx},{cy})\n"
        features_text += f"Bounding Box: x={x}, y={y}, w={w}, h={h}\n\n"

    return features_text


# 🧠 Prediction Function
def predict_image(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    img_norm = img_resized / 255.0
    img_input = img_norm.reshape(1, IMG_SIZE, IMG_SIZE, 1)

    # U-Net prediction
    pred = model.predict(img_input)[0].squeeze()

    # Convert to binary mask
    mask = (pred > 0.2).astype(np.uint8) * 255

    # Convert original to color for overlay
    img_color = cv2.cvtColor(img_resized, cv2.COLOR_GRAY2BGR)

    # 🔴 Overlay RED mask
    red_mask = np.zeros_like(img_color)
    red_mask[:, :, 2] = mask  # Red channel

    overlay = cv2.addWeighted(img_color, 0.7, red_mask, 0.3, 0)

    # Draw bounding boxes
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        cv2.rectangle(overlay, (x, y), (x+w, y+h), (0, 255, 0), 2)

    # Extract features
    features = extract_features(mask)

    # Convert to display
    input_pil = Image.fromarray(img_resized).convert("L").resize((300, 300))
    output_pil = Image.fromarray(overlay).resize((300, 300))

    input_tk = ImageTk.PhotoImage(input_pil)
    output_tk = ImageTk.PhotoImage(output_pil)

    panel_input.config(image=input_tk)
    panel_input.image = input_tk

    panel_output.config(image=output_tk)
    panel_output.image = output_tk

    info_label.config(text=features)


# 📂 Browse
def browse_file():
    file_path = filedialog.askopenfilename()
    if file_path:
        predict_image(file_path)


# Button
btn = tk.Button(root, text="Browse CT Image", command=browse_file)
btn.pack(pady=20)

root.mainloop()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 321ms/step


In [1]:
import tkinter as tk
from tkinter import filedialog
from PIL import Image, ImageTk
import numpy as np
import cv2
from tensorflow.keras.models import load_model

# Load model
model = load_model("unet_pe_model.h5")

IMG_SIZE = 256

# GUI
root = tk.Tk()
root.title("Pulmonary Embolism Detection")
root.geometry("900x500")

panel_input = tk.Label(root)
panel_input.pack(side="left", padx=10)

panel_output = tk.Label(root)
panel_output.pack(side="right", padx=10)

info_label = tk.Label(root, text="", font=("Arial", 12))
info_label.pack(pady=10)


# 🔍 Feature Extraction Function
def extract_features(binary_mask):
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    features_text = ""

    for cnt in contours:
        area = cv2.contourArea(cnt)
        perimeter = cv2.arcLength(cnt, True)

        # Bounding box
        x, y, w, h = cv2.boundingRect(cnt)

        # Centroid
        M = cv2.moments(cnt)
        if M["m00"] != 0:
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
        else:
            cx, cy = 0, 0

        features_text += f"Area: {area:.2f}, Perimeter: {perimeter:.2f}\n"
        features_text += f"Centroid: ({cx},{cy})\n"
        features_text += f"Bounding Box: x={x}, y={y}, w={w}, h={h}\n\n"

    return features_text


# 🧠 Prediction Function
def predict_image(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    img_norm = img_resized / 255.0
    img_input = img_norm.reshape(1, IMG_SIZE, IMG_SIZE, 1)

    # U-Net prediction
    pred = model.predict(img_input)[0].squeeze()

    # Convert to binary mask
    mask = (pred > 0.2).astype(np.uint8) * 255

    # Convert original to color for overlay
    img_color = cv2.cvtColor(img_resized, cv2.COLOR_GRAY2BGR)

    # 🔴 Overlay RED mask
    red_mask = np.zeros_like(img_color)
    red_mask[:, :, 2] = mask  # Red channel

    overlay = cv2.addWeighted(img_color, 0.8, red_mask, 0.3, 0)

    # Draw bounding boxes
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        cv2.rectangle(overlay, (x, y), (x+w, y+h), (0, 255, 0), 2)

    # Extract features
    features = extract_features(mask)

    # Convert to display
    input_pil = Image.fromarray(img_resized).convert("L").resize((300, 300))
    output_pil = Image.fromarray(overlay).resize((300, 300))

    input_tk = ImageTk.PhotoImage(input_pil)
    output_tk = ImageTk.PhotoImage(output_pil)

    panel_input.config(image=input_tk)
    panel_input.image = input_tk

    panel_output.config(image=output_tk)
    panel_output.image = output_tk

    info_label.config(text=features)


# 📂 Browse
def browse_file():
    file_path = filedialog.askopenfilename()
    if file_path:
        predict_image(file_path)


# Button
btn = tk.Button(root, text="Browse CT Image", command=browse_file)
btn.pack(pady=20)

root.mainloop();

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 438ms/step
